In [2]:
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt

print("🚀 HelpDesk AI - Preprocessing Started")

🚀 HelpDesk AI - Preprocessing Started


In [3]:
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print(PROCESSED_DIR)

d:\Development\helpdesk-ai\data\processed


In [4]:
def load_movie_lines(filepath):

    movie_lines = {}

    with open(filepath, encoding="iso-8859-1") as f:

        for line in f:

            parts = line.strip().split(" +++$+++ ")

            if len(parts) == 5:

                movie_lines[parts[0]] = {
                    "character": parts[3],
                    "text": parts[4]
                }

    return movie_lines


def load_conversations(filepath):

    conversations = []

    with open(filepath, encoding="iso-8859-1") as f:

        for line in f:

            parts = line.strip().split(" +++$+++ ")

            if len(parts) == 4:

                ids = (
                    parts[3]
                    .replace("[", "")
                    .replace("]", "")
                    .replace("'", "")
                    .replace(",", "")
                    .split()
                )

                conversations.append(ids)

    return conversations

In [19]:
movie_lines = load_movie_lines(RAW_DIR / "movie_lines.txt")
conversations = load_conversations(RAW_DIR / "movie_conversations.txt")

print(len(movie_lines))
print(len(conversations))

304446
83097


In [20]:
def clean_text(text):

    text = text.lower()

    contractions = {
        "i'm": "i am",
        "he's": "he is",
        "she's": "she is",
        "it's": "it is",
        "that's": "that is",
        "what's": "what is",
        "where's": "where is",
        "who's": "who is",
        "how's": "how is",
        "won't": "will not",
        "can't": "cannot",
        "n't": " not",
        "'re": " are",
        "'ll": " will",
        "'ve": " have",
        "'d": " would",
        "'m": " am"
    }

    for contraction, expansion in contractions.items():
        text = text.replace(contraction, expansion)

    # Keep only letters, numbers and common punctuation
    text = re.sub(r"[^a-zA-Z0-9?.!, ]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [21]:
clean_questions = []
clean_answers = []

for q, a in question_answer_pairs:

    q = clean_text(q)
    a = clean_text(a)

    # Ignore very short sentences
    if len(q.split()) >= 2 and len(a.split()) >= 2:

        clean_questions.append(q)
        clean_answers.append(a)

print("Questions:", len(clean_questions))
print("Answers:", len(clean_answers))

Questions: 190101
Answers: 190101


In [22]:
for i in range(10):

    print("Q:", clean_questions[i])
    print("A:", clean_answers[i])
    print("-" * 60)

Q: can we make this quick? roxanne korrine and andrew barrett are having an incredibly horrendous public break up on the quad. again.
A: well, i thought we would start with pronunciation, if that is okay with you.
------------------------------------------------------------
Q: well, i thought we would start with pronunciation, if that is okay with you.
A: not the hacking and gagging and spitting part. please.
------------------------------------------------------------
Q: not the hacking and gagging and spitting part. please.
A: okay... then how bout we try out some french cuisine. saturday? night?
------------------------------------------------------------
Q: you are asking me out. that is so cute. what is your name again?
A: forget it.
------------------------------------------------------------
Q: the thing is, cameron i am at the mercy of a particularly hideous breed of loser. my sister. i cannot date until she does.
A: seems like she could get a date easy enough...
--------------

In [23]:
MAX_LENGTH = 20

filtered_questions = []
filtered_answers = []

for q, a in zip(clean_questions, clean_answers):

    if len(q.split()) <= MAX_LENGTH and len(a.split()) <= MAX_LENGTH:

        filtered_questions.append(q)
        filtered_answers.append(a)

print("Filtered Questions:", len(filtered_questions))
print("Filtered Answers:", len(filtered_answers))

Filtered Questions: 138230
Filtered Answers: 138230


In [24]:
filtered_answers = [
    "<SOS> " + answer + " <EOS>"
    for answer in filtered_answers
]

In [25]:
for i in range(5):

    print("Q:", filtered_questions[i])
    print("A:", filtered_answers[i])
    print("-" * 60)

Q: well, i thought we would start with pronunciation, if that is okay with you.
A: <SOS> not the hacking and gagging and spitting part. please. <EOS>
------------------------------------------------------------
Q: not the hacking and gagging and spitting part. please.
A: <SOS> okay... then how bout we try out some french cuisine. saturday? night? <EOS>
------------------------------------------------------------
Q: you are asking me out. that is so cute. what is your name again?
A: <SOS> forget it. <EOS>
------------------------------------------------------------
Q: gosh, if only we could find kat a boyfriend...
A: <SOS> let me see what i can do. <EOS>
------------------------------------------------------------
Q: c esc ma tete. this is my head
A: <SOS> right. see? you are ready for the quiz. <EOS>
------------------------------------------------------------
